# Fine-Tuning CampusFlow AI Coach (`Gemma 3 4B Instruct` QLoRA -> GGUF Ollama)

**Kenapa Gemma 3 4B?**
- Ukuran GGUF `Q4_K_M` hanya **~2.5 GB** (hampir separuh dari Qwen 2.5 7B yang `4.7 GB`).
- Bahasa Indonesia natural, kuat mematuhi format JSON & guardrail klinis CampusFlow.

**Dataset yang dibutuhkan (dari `backend/scripts/`):**
- `balanced_dataset_narratives_train_messages.jsonl` (`1000` baris — multi-skenario bervariasi)
- `balanced_dataset_narratives_val_messages.jsonl` (`120` baris — multi-skenario bervariasi)

**Pastikan Runtime Colab memakai GPU (`Runtime` -> `Change runtime type` -> `T4 GPU` / `L4 GPU`).**

In [ ]:
!nvidia-smi
!pip install -q --upgrade unsloth

In [ ]:
import os
import glob

TRAIN_NAME = "balanced_dataset_narratives_train_messages.jsonl"
VAL_NAME = "balanced_dataset_narratives_val_messages.jsonl"

def find_file(filename):
    for pattern in [f"/content/{filename}", f"/content/drive/MyDrive/**/{filename}"]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

# Jika file disimpan di Google Drive, uncomment 2 baris berikut:
# from google.colab import drive
# drive.mount('/content/drive')

train_path = find_file(TRAIN_NAME)
val_path = find_file(VAL_NAME)

if not train_path or not val_path:
    from google.colab import files
    print(f"Silakan upload {TRAIN_NAME} dan {VAL_NAME}...")
    uploaded = files.upload()
    train_path = find_file(TRAIN_NAME)
    val_path = find_file(VAL_NAME)

assert train_path and os.path.exists(train_path), f"{TRAIN_NAME} tidak ditemukan!"
assert val_path and os.path.exists(val_path), f"{VAL_NAME} tidak ditemukan!"
print("Train:", train_path, "| baris:", sum(1 for _ in open(train_path, encoding="utf-8")))
print("Val  :", val_path, "| baris:", sum(1 for _ in open(val_path, encoding="utf-8")))

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/gemma-3-4b-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # Hanya latih teks (hemat VRAM & cepat)
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,                    # Stabil untuk dataset 462 baris (anti-overfit)
    lora_dropout=0,
    bias="none",
    random_state=42,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
print("Model Gemma 3 4B & LoRA siap di:", torch.cuda.get_device_name(0))

In [ ]:
import json
from datasets import load_dataset

def normalize_for_gemma(convo):
    """Gabungkan role system ke awal pesan user pertama agar 100% konsisten dengan Modelfile Ollama."""
    sys_msg = ""
    out = []
    for m in convo:
        role = m["role"]
        content = m["content"]
        if role == "system":
            sys_msg = content
        elif role == "user":
            merged = f"{sys_msg}\n\n{content}" if sys_msg else content
            sys_msg = ""
            out.append({"role": "user", "content": merged})
        elif role == "assistant":
            out.append({"role": "assistant", "content": content})
    return out

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []
    for convo in convos:
        norm = normalize_for_gemma(convo)
        text = tokenizer.apply_chat_template(
            norm, tokenize=False, add_generation_prompt=False
        )
        # Hapus <bos> ganda jika sudah ditambahkan oleh tokenizer
        if text.startswith("<bos>"):
            text = text[len("<bos>"):]
        texts.append(text)
    return {"text": texts}

train_ds = load_dataset("json", data_files=train_path, split="train").shuffle(seed=42).map(
    formatting_prompts_func, batched=True
)
val_ds = load_dataset("json", data_files=val_path, split="train").map(
    formatting_prompts_func, batched=True
)

with open(train_path, encoding="utf-8") as f:
    TRAINING_SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

print("Train rows:", len(train_ds), "| Val rows:", len(val_ds))
print("Contoh 450 karakter pertama:\n", train_ds[0]["text"][:450])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import torch

use_bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.08,
        num_train_epochs=3,
        learning_rate=1e-4,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs_campusflow_gemma3_4b",
        report_to="none",
    ),
)

# WAJIB untuk Gemma: gunakan token <start_of_turn>user dan <start_of_turn>model
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# Verifikasi masking label tidak kosong sebelum training
sample_labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
assert len(sample_labels) > 10, "Label masking gagal! Cek delimiter Gemma."
print("Token respons terdeteksi (baris #0):", len(sample_labels))

stats = trainer.train()
print(stats)

In [ ]:
# Smoke test inferensi sebelum export GGUF
def ask_local(user_json_str):
    msgs = [
        {"role": "user", "content": f"{TRAINING_SYSTEM_PROMPT}\n\n{user_json_str}"}
    ]
    prompt_text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=320,
        do_sample=False,
        use_cache=True,
    )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)

with open(val_path, encoding="utf-8") as f:
    val_rows = [json.loads(line) for line in f if line.strip()]

print("=== Uji Baris Val #1 (Wellbeing) ===")
print(ask_local(val_rows[0]["messages"][1]["content"]))
print("\n=== Uji Baris Val Terakhir (Out-of-Scope) ===")
print(ask_local(val_rows[-1]["messages"][1]["content"]))

In [ ]:
import glob
import os
import shutil

EXPORT_DIR = "campusflow_gemma3_4b_gguf"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Kompatibel dengan semua versi Unsloth (quantization_type vs quantization_method)
try:
    model.save_pretrained_gguf(
        EXPORT_DIR,
        tokenizer,
        quantization_type="Q4_K_M",
    )
except TypeError:
    model.save_pretrained_gguf(
        EXPORT_DIR,
        tokenizer,
        quantization_method="q4_k_m",
    )

gguf_files = sorted(
    [
        p for p in glob.glob("**/*.gguf", recursive=True)
        if "drive/MyDrive" not in p and "f16" not in os.path.basename(p).lower()
    ],
    key=os.path.getmtime,
    reverse=True,
)
if not gguf_files:
    gguf_files = sorted(
        [p for p in glob.glob("**/*.gguf", recursive=True) if "drive/MyDrive" not in p],
        key=os.path.getmtime,
        reverse=True,
    )
assert gguf_files, "File GGUF tidak ditemukan setelah export!"
gguf_path = gguf_files[0]
gguf_basename = os.path.basename(gguf_path)
dest = os.path.join(EXPORT_DIR, gguf_basename)
if os.path.abspath(gguf_path) != os.path.abspath(dest):
    shutil.move(gguf_path, dest)
    gguf_path = dest

# Tulis Modelfile Gemma 3 yang 100% konsisten dengan training + RTX 3050 4GB
modelfile_template = '''FROM ./__GGUF_FILE__
PARAMETER temperature 0
PARAMETER top_p 1
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER num_predict 320
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<eos>"

TEMPLATE """{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if or (eq .Role "user") (eq .Role "system") }}<start_of_turn>user
{{- if and $.System (eq $i 0) }}{{ $.System }}

{{ end }}{{ .Content }}<end_of_turn>
{{ if $last }}<start_of_turn>model
{{ end }}
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}{{ if not $last }}<end_of_turn>
{{ end }}
{{- end }}
{{- end }}"""

SYSTEM """__SYSTEM_PROMPT__"""
'''

modelfile_content = modelfile_template.replace("__GGUF_FILE__", gguf_basename).replace(
    "__SYSTEM_PROMPT__", TRAINING_SYSTEM_PROMPT.replace('"""', '\"\"\"')
)

modelfile_path = os.path.join(EXPORT_DIR, "Modelfile")
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Siap! File GGUF:", gguf_path)
print("Siap! Modelfile:", modelfile_path)

# Jika Google Drive ter-mount, salin otomatis agar mudah didownload:
if os.path.exists("/content/drive/MyDrive"):
    drive_out = "/content/drive/MyDrive/campusflow_gemma3_4b_gguf"
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy2(gguf_path, os.path.join(drive_out, gguf_basename))
    shutil.copy2(modelfile_path, os.path.join(drive_out, "Modelfile"))
    print("Disalin ke Google Drive:", drive_out)